In [1]:
import os, gc, time, json, warnings
from itertools import product

import numpy as np
import pandas as pd
import lightgbm as lgb

warnings.filterwarnings('ignore')
pd.set_option('display.width', 200)

CFG = dict(
    train_path = 'train.csv',
    test_path  = 'test.csv',
    out_path   = 'submission.csv',
    cache      = 'panel_v5.pkl',

    seeds       = [42, 202],
    n_threads   = os.cpu_count(),
    use_leads   = True,

    tune_recency  = True,
    halflife_grid = [None, 1095, 730, 365, 180],

    calibrate      = True,
    calib_min_gain = 0.05,

    blend_with        = ['will model.csv'],
    blend_teammate_w  = 0.5,

    y_min = 2.0,
    y_max = 999.0,
    n_train_expected = 360_954,
    n_test_expected  = 51_063,
    banned_cols = ('current_PM2_5', 'current_PM25', 'PM2_5', 'PM2.5'),
)

print('threads:', CFG['n_threads'], '| leads:', CFG['use_leads'])

threads: 16 | leads: True


In [2]:
POLL = ['PM10', 'SO2', 'NO2', 'CO', 'O3']
MET  = ['TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

WD_DEG = {'N': 0, 'NNE': 22.5, 'NE': 45, 'ENE': 67.5, 'E': 90, 'ESE': 112.5,
          'SE': 135, 'SSE': 157.5, 'S': 180, 'SSW': 202.5, 'SW': 225, 'WSW': 247.5,
          'W': 270, 'WNW': 292.5, 'NW': 315, 'NNW': 337.5}


def shrink(d):
    for c in d.columns:
        if d[c].dtype == np.float64:
            d[c] = d[c].astype(np.float32)
    return d


def check_schema(tr, te):
    stale = sorted({c for c in CFG['banned_cols'] if c in tr.columns or c in te.columns})
    if stale:
        print('!' * 76)
        print(f'STALE DATASET: found {stale}. A lead of this column is an exact target copy.')
        print('Dropping it here, but re-download before submitting.')
        print('!' * 76)
        tr = tr.drop(columns=[c for c in stale if c in tr.columns])
        te = te.drop(columns=[c for c in stale if c in te.columns])
    else:
        print('schema ok: no current-PM2.5 column in either file')

    assert 'PM2_5_next_hour' in tr.columns, 'train has no target column'
    assert 'PM2_5_next_hour' not in te.columns, 'test should not carry the target'
    assert len(tr) == CFG['n_train_expected'], f'train rows {len(tr):,}'
    assert len(te) == CFG['n_test_expected'], f'test rows {len(te):,}'
    assert tr.id.is_unique and te.id.is_unique, 'duplicate ids'
    print(f'train {len(tr):,}  test {len(te):,}  {len(te.columns)} raw predictors')
    return tr, te


def load_panel():
    tr = pd.read_csv(CFG['train_path'], parse_dates=['observation_timestamp'])
    te = pd.read_csv(CFG['test_path'],  parse_dates=['observation_timestamp'])
    tr, te = check_schema(tr, te)

    tr['is_test'] = 0
    te['is_test'] = 1
    te['PM2_5_next_hour'] = np.nan
    df = pd.concat([tr, te], ignore_index=True)
    df['station'] = df.station.astype(str).str.strip()

    ts = pd.date_range(df.observation_timestamp.min(),
                       df.observation_timestamp.max(), freq='h')
    grid = pd.MultiIndex.from_product(
        [sorted(df.station.unique()), ts],
        names=['station', 'observation_timestamp']).to_frame(index=False)

    d = grid.merge(df, on=['station', 'observation_timestamp'], how='left')
    d['present'] = d.id.notna().astype(np.int8)
    d = d.sort_values(['station', 'observation_timestamp']).reset_index(drop=True)

    filled = len(d) - int(d.present.sum())
    print(f'panel {len(d):,} station-hours  ({filled:,} grid holes held open for lag correctness)')
    return d

In [3]:
def add_features(d):
    t = d.observation_timestamp
    d['f_hour']  = t.dt.hour.astype(np.int16)
    d['f_doy']   = t.dt.dayofyear.astype(np.int16)
    d['f_dow']   = t.dt.dayofweek.astype(np.int16)
    d['f_month'] = t.dt.month.astype(np.int16)
    d['t_idx']   = ((t - t.min()).dt.total_seconds() // 3600).astype(np.int32)
    d['hour_sin'] = np.sin(2 * np.pi * d.f_hour / 24)
    d['hour_cos'] = np.cos(2 * np.pi * d.f_hour / 24)
    d['doy_sin']  = np.sin(2 * np.pi * d.f_doy / 365.25)
    d['doy_cos']  = np.cos(2 * np.pi * d.f_doy / 365.25)
    d['is_heating'] = ((d.f_month >= 11) | (d.f_month <= 3)).astype(np.int8)

    rad = np.deg2rad(d.wd.map(WD_DEG))
    d['wd_sin'] = np.sin(rad)
    d['wd_cos'] = np.cos(rad)
    d['wind_u'] = -d.WSPM * np.sin(rad)
    d['wind_v'] = -d.WSPM * np.cos(rad)

    a, b = 17.625, 243.04
    d['RH'] = 100 * np.exp(a * d.DEWP / (b + d.DEWP) - a * d.TEMP / (b + d.TEMP))
    d['dew_dep'] = d.TEMP - d.DEWP
    d['n_missing'] = d[POLL + MET].isna().sum(axis=1).astype(np.int8)

    g, new = d.groupby('station', sort=False), {}
    for L in (1, 2, 3, 4, 6, 8, 12, 18, 24, 36, 48):
        new[f'PM10_lag{L}'] = g.PM10.shift(L)
    for c in ('CO', 'NO2'):
        for L in (1, 2, 3, 6, 12, 24):
            new[f'{c}_lag{L}'] = g[c].shift(L)
    for c in ('SO2', 'O3', 'TEMP', 'PRES', 'DEWP', 'WSPM', 'RH'):
        for L in (1, 3, 6, 24):
            new[f'{c}_lag{L}'] = g[c].shift(L)
    if CFG['use_leads']:
        for c in POLL + MET + ['RH', 'wind_u', 'wind_v', 'dew_dep']:
            new[f'{c}_lead1'] = g[c].shift(-1)
        for c in ('PM10', 'CO', 'NO2'):
            new[f'{c}_lead2'] = g[c].shift(-2)
            new[f'{c}_lead3'] = g[c].shift(-3)
    d = pd.concat([d, shrink(pd.DataFrame(new, index=d.index))], axis=1)
    del new; gc.collect()

    g, new = d.groupby('station', sort=False), {}
    for c in ('PM10', 'CO', 'NO2'):
        for w in (3, 6, 12, 24, 48):
            new[f'{c}_rm{w}'] = g[c].rolling(w, min_periods=2).mean().reset_index(level=0, drop=True)
        new[f'{c}_rsd24'] = g[c].rolling(24, min_periods=4).std().reset_index(level=0, drop=True)
    new['PM10_rmax24'] = g.PM10.rolling(24, min_periods=4).max().reset_index(level=0, drop=True)
    new['PM10_rmin24'] = g.PM10.rolling(24, min_periods=4).min().reset_index(level=0, drop=True)
    for c, ws in [('WSPM', (3, 6, 24)), ('RH', (6, 24)), ('wind_u', (6, 24)),
                  ('wind_v', (6, 24)), ('TEMP', (24,)), ('PRES', (24,))]:
        for w in ws:
            new[f'{c}_rm{w}'] = g[c].rolling(w, min_periods=2).mean().reset_index(level=0, drop=True)
    new['WSPM_rmin24'] = g.WSPM.rolling(24, min_periods=4).min().reset_index(level=0, drop=True)
    for w in (3, 6, 24):
        new[f'RAIN_rs{w}'] = g.RAIN.rolling(w, min_periods=1).sum().reset_index(level=0, drop=True)
    d = pd.concat([d, shrink(pd.DataFrame(new, index=d.index))], axis=1)
    del new; gc.collect()

    new = {}
    for L in (1, 2, 3, 6, 12, 24):
        new[f'PM10_d{L}'] = d.PM10 - d[f'PM10_lag{L}']
    new['PM10_acc'] = (d.PM10 - d.PM10_lag1) - (d.PM10_lag1 - d.PM10_lag2)
    for c in ('CO', 'NO2'):
        for L in (1, 3, 24):
            new[f'{c}_d{L}'] = d[c] - d[f'{c}_lag{L}']
    for c in ('SO2', 'O3', 'TEMP', 'PRES', 'DEWP', 'WSPM'):
        for L in (3, 24):
            new[f'{c}_d{L}'] = d[c] - d[f'{c}_lag{L}']
    if CFG['use_leads']:
        for c in ('PM10', 'CO', 'NO2', 'SO2', 'O3', 'TEMP', 'PRES', 'WSPM', 'RH'):
            new[f'{c}_dlead1'] = d[f'{c}_lead1'] - d[c]
    d = pd.concat([d, shrink(pd.DataFrame(new, index=d.index))], axis=1)
    del new; gc.collect()

    ct, new = d.groupby('observation_timestamp', sort=False), {}
    for c in POLL:
        cm = ct[c].transform('mean')
        new[f'city_{c}'] = cm
        new[f'{c}_dev']  = d[c] - cm
        new[f'{c}_rat']  = d[c] / (cm + 1e-3)
        new[f'log_{c}']  = np.log1p(d[c].clip(lower=0))
    if CFG['use_leads']:
        new['city_PM10_lead1'] = ct['PM10_lead1'].transform('mean')
        new['city_CO_lead1']   = ct['CO_lead1'].transform('mean')
    d = pd.concat([d, shrink(pd.DataFrame(new, index=d.index))], axis=1)
    del new; gc.collect()

    g = d.groupby('station', sort=False)
    new = {
        'city_PM10_lag1': g.city_PM10.shift(1),
        'city_PM10_lag3': g.city_PM10.shift(3),
        'city_PM10_rm24': g.city_PM10.rolling(24, min_periods=2).mean().reset_index(level=0, drop=True),
        'city_CO_lag1':   g.city_CO.shift(1),
        'CO_over_PM10':   d.CO / (d.PM10 + 1.0),
        'NO2_over_O3':    d.NO2 / (d.O3 + 1.0),
        'SO2_over_PM10':  d.SO2 / (d.PM10 + 1.0),
        'PM10_vent':      d.PM10 / (d.WSPM + 0.1),
        'PM10_x_RH':      d.PM10 * d.RH / 100,
        'PM10_x_RH2':     d.PM10 * (d.RH / 100) ** 2,
        'PM10_hygro':     d.PM10 / np.maximum(0.02, 1.02 - d.RH / 100),
        'PM10_x_RH24':    d.PM10 * d.RH_rm24 / 100,
        'PM10lag1_x_RH':  d.PM10_lag1 * d.RH_lag1 / 100,
        'cityPM10_x_RH':  d.city_PM10 * d.RH / 100,
        'CO_x_RH':        d.CO * d.RH / 100,
        'PM10_x_dewdep':  d.PM10 * d.dew_dep,
    }
    if CFG['use_leads']:
        new.update({
            'PM10lead_x_RHlead': d.PM10_lead1 * d.RH_lead1 / 100,
            'PM10lead_hygro':    d.PM10_lead1 / np.maximum(0.02, 1.02 - d.RH_lead1 / 100),
            'log_PM10_lead1':    np.log1p(d.PM10_lead1.clip(lower=0)),
            'log_CO_lead1':      np.log1p(d.CO_lead1.clip(lower=0)),
            'CO_over_PM10_lead': d.CO_lead1 / (d.PM10_lead1 + 1.0),
            'PM10lead_vent':     d.PM10_lead1 / (d.WSPM_lead1 + 0.1),
            'PM10lead_dev':      d.PM10_lead1 - d.city_PM10_lead1,
            'city_PM10_dlead1':  d.city_PM10_lead1 - d.city_PM10,
            'city_CO_dlead1':    d.city_CO_lead1 - d.city_CO,
        })
    new['city_PM10_d1'] = d.city_PM10 - new['city_PM10_lag1']
    new['city_PM10_d3'] = d.city_PM10 - new['city_PM10_lag3']
    d = pd.concat([d, shrink(pd.DataFrame(new, index=d.index))], axis=1)
    del new; gc.collect()

    d = d[d.present == 1].reset_index(drop=True)
    d.drop(columns=['present', 'wd', 'No'], errors='ignore', inplace=True)
    d['station'] = d.station.astype('category')
    return shrink(d)

In [4]:
t0 = time.time()
if os.path.exists(CFG['cache']):
    data = pd.read_pickle(CFG['cache'])
    print(f'loaded cache {CFG["cache"]}')
else:
    data = add_features(load_panel())
    data.to_pickle(CFG['cache'])
gc.collect()
print(f'{data.shape}  {time.time()-t0:.0f}s  {data.memory_usage(deep=True).sum()/1e9:.2f} GB')

loaded cache panel_v5.pkl
(412017, 225)  0s  0.37 GB


In [5]:
NON_FEATURES = {'observation_timestamp', 'PM2_5_next_hour', 'is_test', 'id',
                'year', 'month', 'day', 'hour'}
FEATS = [c for c in data.columns if c not in NON_FEATURES]
LEAD_FEATS = [c for c in FEATS if 'lead' in c]
CATS = ['station']

train = data[data.is_test == 0].reset_index(drop=True)
test  = data[data.is_test == 1].reset_index(drop=True)
y     = train.PM2_5_next_hour.values.astype(np.float64)
ts    = train.observation_timestamp
age_days = (train.t_idx.max() - train.t_idx.values) / 24.0

print(f'{len(FEATS)} features ({len(LEAD_FEATS)} lead-derived)  '
      f'train {len(train):,}  test {len(test):,}')
assert 'station' in FEATS, 'station must reach the model'


def rmse(a, p):
    return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(p)) ** 2)))


def season_folds():
    out = []
    for yr in sorted(ts.dt.year.unique()):
        cut, end = pd.Timestamp(f'{yr}-09-01'), pd.Timestamp(f'{yr+1}-03-01')
        tr = np.where(ts < cut)[0]
        va = np.where((ts >= cut) & (ts < end))[0]
        if len(tr) > 30_000 and len(va) > 5_000:
            out.append((f'S{yr}', tr, va))
    return out


FOLDS = season_folds()
for nm, tr, va in FOLDS:
    print(f'  {nm}  fit {len(tr):>7,}   valid {len(va):>6,}')

217 features (40 lead-derived)  train 360,954  test 51,063
  S2013  fit  51,925   valid 51,442
  S2014  fit 155,190   valid 50,799
  S2015  fit 257,927   valid 51,349


In [6]:
BASE = dict(objective='regression', metric='rmse', bagging_fraction=0.8, bagging_freq=1,
            max_bin=255, num_threads=CFG['n_threads'], force_col_wise=True, verbosity=-1)
LR = 0.04

SPECS = [
    ('lo_L2',  dict(BASE, learning_rate=LR, num_leaves=63,  min_data_in_leaf=120,
                    feature_fraction=0.50, lambda_l2=20.0), False),
    ('hi_L2',  dict(BASE, learning_rate=LR, num_leaves=160, min_data_in_leaf=40,
                    feature_fraction=0.65, lambda_l2=6.0),  False),
    ('deep',   dict(BASE, learning_rate=LR, num_leaves=320, min_data_in_leaf=25,
                    feature_fraction=0.35, lambda_l1=0.5, lambda_l2=8.0), False),
    ('lo_log', dict(BASE, learning_rate=LR, num_leaves=63,  min_data_in_leaf=120,
                    feature_fraction=0.50, lambda_l2=20.0), True),
]
print('specs:', [s[0] for s in SPECS])


def fit_lgb(tr_idx, va_idx, w, params, log, rounds=6000, es=150, seed=42):
    p = dict(params, seed=seed, bagging_seed=seed + 1, feature_fraction_seed=seed + 2)
    yy = np.log1p(y) if log else y
    dtr = lgb.Dataset(train.iloc[tr_idx][FEATS], yy[tr_idx],
                      weight=w[tr_idx], categorical_feature=CATS)
    dva = lgb.Dataset(train.iloc[va_idx][FEATS], yy[va_idx], reference=dtr)
    m = lgb.train(p, dtr, rounds, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(es, verbose=False)])
    pred = m.predict(train.iloc[va_idx][FEATS], num_iteration=m.best_iteration)
    if log:
        pred = np.expm1(pred)
    return m, np.clip(pred, CFG['y_min'], CFG['y_max'])

specs: ['lo_L2', 'hi_L2', 'deep', 'lo_log']


In [7]:
def weights(hl):
    return np.ones(len(train)) if hl is None else 0.5 ** (age_days / hl)


HALFLIFE = None
if CFG['tune_recency']:
    _, tr_i, va_i = FOLDS[-1]
    probe = dict(SPECS[0][1], learning_rate=0.12)
    scores = {}
    for hl in CFG['halflife_grid']:
        m, p = fit_lgb(tr_i, va_i, weights(hl), probe, False, rounds=900, es=60)
        scores[hl] = rmse(y[va_i], p)
        print(f'  half-life {str(hl):>5} -> RMSE {scores[hl]:.4f}  (iters {m.best_iteration})')
        del m; gc.collect()
    HALFLIFE = min(scores, key=scores.get)
    print(f'chosen half-life: {HALFLIFE}')

W = weights(HALFLIFE)
print(f'weight range {W.min():.4f} - {W.max():.4f}')

  half-life  None -> RMSE 23.0928  (iters 181)
  half-life  1095 -> RMSE 21.7949  (iters 329)
  half-life   730 -> RMSE 22.5154  (iters 197)
  half-life   365 -> RMSE 22.9070  (iters 264)
  half-life   180 -> RMSE 26.0343  (iters 320)
chosen half-life: 1095
weight range 0.4448 - 1.0000


In [8]:
oof = np.full((len(train), len(SPECS)), np.nan)
rounds_for = {}

for si, (name, params, log) in enumerate(SPECS):
    print(f'== {name} ==')
    iters, sizes = [], []
    for nm, tr_i, va_i in FOLDS:
        t0 = time.time()
        m, p = fit_lgb(tr_i, va_i, W, params, log)
        oof[va_i, si] = p
        iters.append(m.best_iteration); sizes.append(len(tr_i))
        print(f'  {nm}: RMSE {rmse(y[va_i], p):.4f}  iters {m.best_iteration}  '
              f'({time.time()-t0:.0f}s)')
        del m; gc.collect()
    rounds_for[name] = int(np.average(iters, weights=sizes))
    print(f'  size-weighted rounds: {rounds_for[name]}')

mask = ~np.isnan(oof).any(axis=1)
OP, OY = oof[mask], y[mask]
for si, (name, _, _) in enumerate(SPECS):
    print(f'{name:7s} pooled OOF RMSE {rmse(OY, OP[:, si]):.4f}')

== lo_L2 ==
  S2013: RMSE 27.9313  iters 184  (4s)


KeyboardInterrupt: 

In [ ]:
grid = np.arange(0, 1.0001, 0.05)
best_w, best_e = None, np.inf
for combo in product(grid, repeat=len(SPECS) - 1):
    if sum(combo) > 1.0001:
        continue
    w = np.array(list(combo) + [1 - sum(combo)])
    e = rmse(OY, OP @ w)
    if e < best_e:
        best_w, best_e = w, e

print('blend:', {s[0]: round(float(v), 2) for s, v in zip(SPECS, best_w)})
print(f'blended OOF RMSE {best_e:.4f}   '
      f'(uniform average would be {rmse(OY, OP.mean(1)):.4f})')
blend_oof = OP @ best_w

In [ ]:
B0, B1 = 0.0, 1.0
if CFG['calibrate'] and len(FOLDS) >= 2:
    pos = {orig: new for new, orig in enumerate(np.where(mask)[0])}
    gains = []
    for k in range(len(FOLDS)):
        other = np.array([pos[x] for j in range(len(FOLDS)) if j != k
                          for x in FOLDS[j][2] if x in pos])
        cur = np.array([pos[x] for x in FOLDS[k][2] if x in pos])
        if len(other) == 0 or len(cur) == 0:
            continue
        b1, b0 = np.polyfit(blend_oof[other], OY[other], 1)
        before = rmse(OY[cur], blend_oof[cur])
        after = rmse(OY[cur], np.clip(b0 + b1 * blend_oof[cur], CFG['y_min'], CFG['y_max']))
        gains.append(before - after)
        print(f'  {FOLDS[k][0]}: b0 {b0:+.2f} b1 {b1:.3f}   {before:.4f} -> {after:.4f}')
    if gains and np.mean(gains) > CFG['calib_min_gain'] and gains[-1] > 0:
        B1, B0 = np.polyfit(blend_oof, OY, 1)
        print(f'  APPLIED: pred -> {B0:+.3f} + {B1:.4f} * pred  (mean gain {np.mean(gains):.3f})')
    else:
        print(f'  DISCARDED: does not transfer out of fold '
              f'(mean gain {np.mean(gains) if gains else 0:.3f})')

In [ ]:
SCALE = len(train) / len(FOLDS[-1][1])
assert SCALE >= 1.0, 'refit uses more data than any fold, so rounds must scale up'
print(f'refitting on all {len(train):,} rows   round multiplier {SCALE:.2f}')

plan = [(name, seed, max(50, int(rounds_for[name] * SCALE)))
        for name, _, _ in SPECS for seed in CFG['seeds']]
print(pd.DataFrame(plan, columns=['spec', 'seed', 'rounds']).to_string(index=False))

spec_preds = {}
for name, params, log in SPECS:
    runs = []
    for seed in CFG['seeds']:
        t0 = time.time()
        p = dict(params, seed=seed, bagging_seed=seed, feature_fraction_seed=seed)
        yy = np.log1p(y) if log else y
        m = lgb.train(p, lgb.Dataset(train[FEATS], yy, weight=W, categorical_feature=CATS),
                      max(50, int(rounds_for[name] * SCALE)))
        pr = m.predict(test[FEATS])
        runs.append(np.expm1(pr) if log else pr)
        print(f'  {name} seed {seed}  ({time.time()-t0:.0f}s)')
        if not log:
            last_model = m
        del m; gc.collect()
    spec_preds[name] = np.clip(np.mean(runs, axis=0), CFG['y_min'], CFG['y_max'])

own = np.column_stack([spec_preds[s[0]] for s in SPECS]) @ best_w
own = np.clip(B0 + B1 * own, CFG['y_min'], CFG['y_max'])
print(f'\nown blend  mean {own.mean():.2f}  std {own.std():.2f}  '
      f'min {own.min():.1f}  max {own.max():.1f}')

In [ ]:
parts, labels = [own], ['own']
for path in CFG['blend_with']:
    if not os.path.exists(path):
        print(f'  skipping {path!r} (not found)')
        continue
    ext = pd.read_csv(path)
    assert {'id', 'PM2_5_next_hour'} <= set(ext.columns), f'{path} has unexpected columns'
    aligned = ext.set_index('id').reindex(test.id.values)['PM2_5_next_hour']
    if aligned.isna().any():
        print(f'  skipping {path!r} ({int(aligned.isna().sum())} ids missing)')
        continue
    parts.append(aligned.values)
    labels.append(path)
    print(f'  blending in {path!r}  corr with own {np.corrcoef(own, aligned.values)[0,1]:.4f}')

if len(parts) == 1:
    final = own
    print('no teammate files blended - shipping own predictions')
else:
    tw = CFG['blend_teammate_w']
    others = np.mean(parts[1:], axis=0)
    final = (1 - tw) * own + tw * others
    print(f'\nfinal = {1-tw:.2f} * own + {tw:.2f} * mean({labels[1:]})')

final = np.clip(final, CFG['y_min'], CFG['y_max'])

In [ ]:
sub = pd.DataFrame({'id': test.id.values, 'PM2_5_next_hour': final})
assert len(sub) == CFG['n_test_expected'], f'expected 51,063 rows, got {len(sub):,}'
assert sub.id.is_unique and sub.id.notna().all(), 'id problem'
assert sub.PM2_5_next_hour.notna().all(), 'NaNs in predictions'
assert sub.PM2_5_next_hour.std() > 1, 'degenerate predictions'

if os.path.exists('sample_submission.csv'):
    ss = pd.read_csv('sample_submission.csv', usecols=['id'])
    assert set(sub.id) == set(ss.id), 'ids do not match sample_submission'
    print(f'ids match sample_submission ({len(ss):,})')

sub.to_csv(CFG['out_path'], index=False, float_format='%.6f')
print(sub.head())
print(f'\nwrote {CFG["out_path"]}  ({len(sub):,} rows)  '
      f'half-life={HALFLIFE}  leads={CFG["use_leads"]}')
print(f'mean {final.mean():.2f}  std {final.std():.2f}   '
      f'(train mean 78.04, train Sep-Feb mean 87.8)')

In [ ]:
diag = pd.DataFrame({
    'y': OY,
    'p': np.clip(B0 + B1 * blend_oof, CFG['y_min'], CFG['y_max']),
    'month': train.observation_timestamp.values[mask],
    'station': train.station.values[mask],
})
diag['err'] = diag.p - diag.y
diag['month'] = pd.Series(pd.to_datetime(diag.month)).dt.to_period('M')
diag['band'] = pd.cut(diag.y, [0, 35, 75, 150, 250, 2000],
                      labels=['0-35', '35-75', '75-150', '150-250', '250+'])

print(f'OOF bias {diag.err.mean():+.3f}   RMSE {rmse(diag.y, diag.p):.4f}\n')

agg = lambda g: pd.Series({'n': len(g), 'rmse': rmse(g.y, g.p), 'bias': g.err.mean()})
print(diag.groupby('band', observed=True).apply(agg, include_groups=False).round(2).to_string())
print()
print(diag.groupby('station', observed=True).apply(agg, include_groups=False)
          .round(2).sort_values('bias').to_string())

In [ ]:
imp = pd.Series(last_model.feature_importance('gain'), index=FEATS).sort_values(ascending=False)
print('top 25 features by gain:')
print(imp.head(25).to_string())
lead_share = imp[[c for c in imp.index if 'lead' in c]].sum() / imp.sum()
print(f'\nlead-derived share of total gain: {lead_share:.1%}   '
      f'<- your exposure if the organisers patch leads')